In [1]:
import os
import glob
import numpy as np
from skimage.segmentation import watershed, mark_boundaries
from skimage.util import img_as_float
from skimage import io, color
from skimage.measure import label
from skimage.filters import sobel # 導入 sobel 濾波器來計算梯度

def generate_pseudolabels_watershed(image_path, output_dir, vis_dir):
    """
    為單張2D彩色圖片生成superpixel pseudolabels (使用 Watershed 演算法)。

    Args:
        image_path (str): 輸入圖片的路徑。
        output_dir (str): 儲存pseudolabel的路徑。
        vis_dir (str): 儲存視覺化結果的路徑。
    """
    # --- 1. 參數設定 ---
    # 控制超像素的數量。值越大，生成的超像素越小、越密集。
    # 這不是一個精確的數字，而是初始標記點的數量。
    N_SEGMENTS = 400
    
    # 控制邊界的平滑程度和緊湊性。值越大，超像素越趨近於圓形。
    COMPACTNESS = 0.01

    # --- 讀取和預處理影像 ---
    try:
        image = io.imread(image_path)
    except Exception as e:
        print(f"Error reading {image_path}: {e}")
        return

    if image.shape[-1] == 4:
        image = image[..., :3]
    
    image_float = img_as_float(image)

    # --- 2. 使用 Watershed 演算法生成 Superpixels ---
    # a. 計算梯度圖：分水嶺演算法在梯度圖上效果最好，因為梯度高的區域是天然的“山脊”屏障。
    gradient = sobel(color.rgb2gray(image_float))
    
    # b. 執行分水嶺演算法
    #    - `markers`: 初始的“洪水”源頭。我們在圖像上均勻地放置 N_SEGMENTS 個標記點。
    #    - `compactness`: 引入緊湊性約束，防止生成過於奇形怪狀的區域。
    segments = watershed(gradient, markers=N_SEGMENTS, compactness=COMPACTNESS)
    
    # 分水嶺的標籤從1開始，所以不需要 +1

    # --- 3. (可選) 移除背景區域的超級像素 ---
    gray_image = color.rgb2gray(image_float)
    foreground_mask = gray_image > 0.1
    segments_masked = segments * foreground_mask

    # --- 4. 重新標記以確保標籤連續 ---
    final_labels, num_labels = label(segments_masked, background=0, return_num=True)
    
    # --- 5. 儲存 Pseudolabel 結果 ---
    base_name = os.path.splitext(os.path.basename(image_path))[0]
    output_filename = os.path.join(output_dir, f"{base_name}_label.png")
    io.imsave(output_filename, final_labels.astype(np.uint16))

    # --- 6. 產生並儲存視覺化結果 ---
    vis_image = mark_boundaries(image_float, final_labels)
    vis_filename = os.path.join(vis_dir, f"{base_name}_vis.png")
    io.imsave(vis_filename, (vis_image * 255).astype(np.uint8))
    
    print(f"Processed {base_name} successfully, generated {num_labels} superpixels using Watershed.")


def main():
    # --- 設定您的路徑 ---
    INPUT_DIR = '/home/U116med/data/polyp/TrainDataset/images' 
    OUTPUT_DIR = './polypdata/pseudolabels_watershed/' # 使用新資料夾
    VISUALIZATION_DIR = './polypdata/visualizations_watershed/'

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.makedirs(VISUALIZATION_DIR, exist_ok=True)

    image_paths = glob.glob(os.path.join(INPUT_DIR, '*.png'))
    image_paths.extend(glob.glob(os.path.join(INPUT_DIR, '*.jpg')))
    
    if not image_paths:
        print(f"Error: No images found in {INPUT_DIR}. Please check the path.")
        return

    print(f"Found {len(image_paths)} images. Starting processing with Watershed...")
    
    for path in image_paths:
        generate_pseudolabels_watershed(path, OUTPUT_DIR, VISUALIZATION_DIR)
        
    print("All images processed.")


if __name__ == '__main__':
    main()

Found 1450 images. Starting processing with Watershed...
Processed 1 successfully, generated 334 superpixels using Watershed.


/tmp/ipykernel_519979/1552822998.py:61: UserWarning: ./polypdata/pseudolabels_watershed/1_label.png is a low contrast image
  io.imsave(output_filename, final_labels.astype(np.uint16))
/tmp/ipykernel_519979/1552822998.py:61: UserWarning: ./polypdata/pseudolabels_watershed/10_label.png is a low contrast image
  io.imsave(output_filename, final_labels.astype(np.uint16))


Processed 10 successfully, generated 348 superpixels using Watershed.
Processed 101 successfully, generated 312 superpixels using Watershed.
Processed 102 successfully, generated 311 superpixels using Watershed.


/tmp/ipykernel_519979/1552822998.py:61: UserWarning: ./polypdata/pseudolabels_watershed/101_label.png is a low contrast image
  io.imsave(output_filename, final_labels.astype(np.uint16))
/tmp/ipykernel_519979/1552822998.py:61: UserWarning: ./polypdata/pseudolabels_watershed/102_label.png is a low contrast image
  io.imsave(output_filename, final_labels.astype(np.uint16))
/tmp/ipykernel_519979/1552822998.py:61: UserWarning: ./polypdata/pseudolabels_watershed/103_label.png is a low contrast image
  io.imsave(output_filename, final_labels.astype(np.uint16))


Processed 103 successfully, generated 312 superpixels using Watershed.
Processed 104 successfully, generated 314 superpixels using Watershed.
Processed 105 successfully, generated 318 superpixels using Watershed.


...

/tmp/ipykernel_519979/1552822998.py:61: UserWarning: ./polypdata/pseudolabels_watershed/ck2bxskgxxzfv08386xkqtqdy_label.png is a low contrast image
  io.imsave(output_filename, final_labels.astype(np.uint16))


Processed ck2bxskgxxzfv08386xkqtqdy successfully, generated 368 superpixels using Watershed.


/tmp/ipykernel_519979/1552822998.py:61: UserWarning: ./polypdata/pseudolabels_watershed/ck2bxw18mmz1k0725litqq2mc_label.png is a low contrast image
  io.imsave(output_filename, final_labels.astype(np.uint16))


Processed ck2bxw18mmz1k0725litqq2mc successfully, generated 399 superpixels using Watershed.
Processed ck2da7fwcjfis07218r1rvm95 successfully, generated 482 superpixels using Watershed.
All images processed.


/tmp/ipykernel_519979/1552822998.py:61: UserWarning: ./polypdata/pseudolabels_watershed/ck2da7fwcjfis07218r1rvm95_label.png is a low contrast image
  io.imsave(output_filename, final_labels.astype(np.uint16))
